# 🟡 Solution: Top-k Sampling

### Exercise contract
Accept logits of shape `(vocab_size,)` or `(B, vocab_size)` and return an integer token-index tensor of shape `()` or `(B,)`, respectively. Sample each row independently. Any algorithm sampling the specified categorical distribution is valid; the judge checks approximate frequencies and allowed tokens, **not identical draws for a fixed seed**.

Assume integer `k >= 1`; cap `k` at vocabulary size. If logits tie at the cutoff, any selection of k highest-scoring tokens is valid.

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def top_k_sample(logits: torch.Tensor, k: int) -> torch.Tensor:
    squeeze = logits.dim() == 1
    if squeeze:
        logits = logits.unsqueeze(0)

    k = min(k, logits.shape[-1])
    top_values, top_indices = torch.topk(logits, k=k, dim=-1)
    masked = torch.full_like(logits, float("-inf"))
    masked.scatter_(-1, top_indices, top_values)
    probs = torch.softmax(masked, dim=-1)
    samples = torch.multinomial(probs, num_samples=1).squeeze(-1)
    return samples.squeeze(0) if squeeze else samples


In [ ]:
# Verify
torch.manual_seed(0)
logits = torch.tensor([[5.0, 4.0, 1.0, -2.0]])
print(top_k_sample(logits, k=2))
